# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thiru-glitch-cmd/flyrank-ml-thirumalai-/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
import pandas as pd

# Unit of analysis: One row represents a unique search query-result pair observed daily.
# Time window: Data covers the last 30 days, aggregated daily.

# Placeholder for verification code:
# For example, check the number of unique query-result pairs per day
# and the date range of the dataset.
# df.groupby('date_column').size().min() # Check for daily data
# df['date_column'].min(), df['date_column'].max() # Check date range

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

This section is crucial for validating the claims made in Section 1 (Unit of Analysis + Time Window) and ensuring the quality of the fields defined in Section 2. We'll use queries to check:

*   **Grain:** Verify that each row truly represents a unique daily search query-result pair.
*   **Counts:** Ensure consistent daily data availability within the specified time window.
*   **Missing Values:** Identify any significant missing data in critical feature or label columns.
*   **Windows:** Confirm the date range and daily aggregation.


Here's a breakdown of potential fields based on the unit of analysis (unique search query-result pair observed daily) and their categorization:

*   **Features:** Variables used as input to predict the label.
*   **Labels:** The target variable we want to predict or optimize.
*   **Context:** Data that provides additional information but is not directly used as a feature or label, or might be used for filtering/grouping.
*   **Excluded:** Fields not used in the analysis, often due to redundancy, privacy concerns, or lack of predictive power. Each excluded field needs a 'why'.


In [2]:
# Assuming a dataset with fields related to search query results.
# Replace these with your actual column names and categorize them accordingly.

features = [
    'query_text',          # The actual search query entered by the user
    'result_url',          # The URL of the search result displayed
    'result_position',     # The rank or position of the result on the search page
    'device_type',         # E.g., 'desktop', 'mobile', 'tablet'
    'geo_country',         # Country from which the search originated
    'query_length',        # Length of the search query
    'query_embedding',     # Vector representation of the query (if available)
    'result_embedding',    # Vector representation of the result content (if available)
    'query_previous_count',# Number of times this query was seen in the last X days
    'user_segment'         # User segmentation (e.g., new_user, returning_user)
]

labels = [
    'clicks',              # Number of clicks on this specific query-result pair
    'impressions',         # Number of times this specific query-result pair was shown
    'conversion'           # Binary indicator if the user converted after clicking
]

context = [
    'date',                # Date of the observation (used for time window, not directly as feature)
    'query_id',            # Unique identifier for the search query (for grouping/tracking)
    'session_id',          # User session identifier
    'user_id'              # Unique identifier for the user
]

excluded = [
    ('internal_log_id', 'Internal system identifier, irrelevant for model training or interpretation.'),
    ('raw_search_log', 'Contains sensitive or unstructured data not suitable for direct use.'),
    ('server_ip_address', 'Privacy concerns and typically not a predictive feature.')
]

print("--- Field Categorization ---")
print(f"Features: {features}")
print(f"Labels: {labels}")
print(f"Context: {context}")
print("Excluded:")
for field, reason in excluded:
    print(f"  - {field}: {reason}")


--- Field Categorization ---
Features: ['query_text', 'result_url', 'result_position', 'device_type', 'geo_country', 'query_length', 'query_embedding', 'result_embedding', 'query_previous_count', 'user_segment']
Labels: ['clicks', 'impressions', 'conversion']
Context: ['date', 'query_id', 'session_id', 'user_id']
Excluded:
  - internal_log_id: Internal system identifier, irrelevant for model training or interpretation.
  - raw_search_log: Contains sensitive or unstructured data not suitable for direct use.
  - server_ip_address: Privacy concerns and typically not a predictive feature.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
import pandas as pd
import numpy as np
from datetime import date, timedelta

# Placeholder: Assuming 'df' is your DataFrame loaded with the data.
# You would typically load your data here, e.g., df = pd.read_csv('your_data.csv')

# Create a dummy DataFrame for demonstration purposes
# Simulating 30 days of data, with 100 query-result pairs per day

start_date = date.today() - timedelta(days=29) # Last 30 days
dates = [start_date + timedelta(days=i) for i in range(30)]

data = []
for d in dates:
    for i in range(100):
        data.append({
            'date': d,
            'query_id': f'query_{np.random.randint(1, 50)}',
            'query_text': f'search term {np.random.randint(1, 20)}',
            'result_url': f'http://example.com/result_{np.random.randint(1, 15)}',
            'result_position': np.random.randint(1, 10),
            'device_type': np.random.choice(['desktop', 'mobile']),
            'geo_country': np.random.choice(['US', 'CA', 'GB']),
            'query_length': np.random.randint(5, 20),
            'query_embedding': np.random.rand(5).tolist(),
            'result_embedding': np.random.rand(5).tolist(),
            'query_previous_count': np.random.randint(0, 100),
            'user_segment': np.random.choice(['new_user', 'returning_user']),
            'clicks': np.random.randint(0, 5),
            'impressions': np.random.randint(1, 10),
            'conversion': np.random.choice([0, 1], p=[0.95, 0.05]),
            'session_id': f'session_{np.random.randint(1000, 2000)}',
            'user_id': f'user_{np.random.randint(100, 500)}',
            'internal_log_id': f'log_{np.random.randint(10000, 99999)}'
        })

df = pd.DataFrame(data)
# Ensure 'date' column is datetime type for verification steps
df['date'] = pd.to_datetime(df['date'])


# --- 3.1 Verify Grain: One row = one unique search query-result pair observed daily ---
# Assuming 'date', 'query_id', and 'result_url' uniquely identify a query-result pair daily.
# If your data has a different granularity, adjust the columns used for uniqueness.
# For example, if 'query_text' and 'result_url' are sufficient and unique after aggregation.

print("--- Grain Verification ---")
# For the dummy data, we expect some duplicates due to random generation
# Adjust subset based on your actual unique grain definition
duplicate_rows = df[df.duplicated(subset=['date', 'query_id', 'result_url'], keep=False)]
if not duplicate_rows.empty:
    print(f"WARNING: Found {len(duplicate_rows)} duplicate rows based on (date, query_id, result_url). This violates the daily unique grain.")
    print("Sample duplicates:")
    print(duplicate_rows.head())
else:
    print("Grain verified: Each row appears to be a unique (date, query_id, result_url) combination.")


# --- 3.2 Verify Counts: Ensure data for each day within the time window ---
print("\n--- Counts Verification ---")
# 'date' column is already datetime from dummy data creation

daily_counts = df.groupby('date').size()
print("Daily row counts:")
print(daily_counts.sort_index().head())
print(daily_counts.sort_index().tail())

# Check for minimum daily count to ensure no empty days
min_daily_count = daily_counts.min()
print(f"Minimum daily row count: {min_daily_count}")
if min_daily_count == 0:
    print("WARNING: Some days have 0 rows. Investigate data completeness.")


# --- 3.3 Verify Missing Values for Key Fields ---
print("\n--- Missing Values Verification ---")
# 'features' and 'labels' lists are available from previous cell's kernel state
key_fields_to_check = features + labels # Combine features and labels for critical check

missing_data = df[key_fields_to_check].isnull().sum()
missing_data = missing_data[missing_data > 0]

if not missing_data.empty:
    print("WARNING: Missing values found in critical fields:")
    print(missing_data.sort_values(ascending=False))
else:
    print("No significant missing values found in critical feature and label fields.")


# --- 3.4 Verify Time Window (last 30 days) and Daily Aggregation ---
print("\n--- Time Window Verification ---")
data_start_date = df['date'].min()
data_end_date = df['date'].max()

print(f"Data covers date range from {data_start_date.strftime('%Y-%m-%d')} to {data_end_date.strftime('%Y-%m-%d')}")

# Check if the range roughly matches 30 days
days_in_data = (data_end_date - data_start_date).days
print(f"Total days observed in data: {days_in_data + 1} days") # +1 to include both start and end day

if days_in_data < 29 or days_in_data > 31:
    print(f"WARNING: The data window ({days_in_data + 1} days) does not exactly match the expected 30 days. Investigate.")
else:
    print("Time window approximately matches the last 30 days.")

# Check for daily aggregation by looking at unique time components other than date
# This is a basic check; a more robust check might involve aggregating and then verifying.
if df['date'].dt.time.nunique() > 1:
    print("WARNING: 'date' column contains time components, suggesting data might not be strictly daily aggregated.")
else:
    print("Data appears to be aggregated daily (no time component in 'date' column).")


--- Grain Verification ---
Sample duplicates:
         date  query_id      query_text                    result_url  \
13 2026-06-23  query_38  search term 10   http://example.com/result_1   
15 2026-06-23  query_34  search term 16   http://example.com/result_7   
16 2026-06-23  query_34  search term 13   http://example.com/result_7   
18 2026-06-23  query_14  search term 13  http://example.com/result_12   
21 2026-06-23  query_49  search term 16   http://example.com/result_9   

    result_position device_type geo_country  query_length  \
13                8     desktop          GB             8   
15                3      mobile          CA             9   
16                8      mobile          GB             7   
18                3      mobile          CA             6   
21                4      mobile          US             8   

                                      query_embedding  \
13  [0.5547264405461194, 0.9652197786086802, 0.657...   
15  [0.11013581334412748, 0.625187

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Even with a well-defined data contract, every dataset has inherent limitations. Understanding these 'data limits' is crucial for properly interpreting model results and preventing misapplication. Here are some examples of what this search intelligence data, as defined, *cannot* tell us:

*   **Unbalanced History**: If certain queries or results appear very infrequently in the history (e.g., new products, seasonal searches), the model trained on this data might perform poorly on these edge cases due to lack of sufficient examples.
*   **Attribution beyond first click**: Our 'clicks' and 'conversion' labels are tied to the specific query-result pair. This data alone cannot tell us about multi-touch attribution (e.g., if a user saw this result, then later found the product via a different path) or the influence of other marketing channels.
*   **User intent not captured by query**: While `query_text` is a feature, subtle shifts in user intent (e.g., informational vs. transactional) might not be fully captured, especially for ambiguous queries. The data doesn't contain direct user feedback on intent.
*   **Search Engine Results Page (SERP) layout changes**: If the SERP layout or advertising slots change frequently, the 'result_position' might have different meanings or impacts over time, which isn't explicitly modeled here.
*   **Non-search user behavior**: The data focuses on search interactions. It doesn't provide insights into user behavior originating from direct navigation, social media, or other referral sources.
*   **Data Latency/Freshness**: The contract specifies 'last 30 days'. We cannot infer anything about real-time trends or how quickly new data becomes available or stale.
*   **Data quality issues outside of defined checks**: While we checked for missing values and grain, there could be other quality issues (e.g., incorrect `geo_country` assignments, spam queries) that are not detectable by these high-level contract queries.


In [5]:
# This section is primarily for qualitative analysis and documentation.
# No executable code is typically required here, but we can add a placeholder
# to affirm understanding of data limits.

print("Acknowledged and understood the potential limitations and blind spots of the data contract.")
print("These limits will be considered when interpreting model performance and business insights.")

Acknowledged and understood the potential limitations and blind spots of the data contract.
These limits will be considered when interpreting model performance and business insights.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.